## Imports

In [ ]:
import os
import polars as pl
from tqdm import tqdm
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import random
import numpy as np
import pickle
from datasets import load_dataset
from dynabatch import dynabatch_sampler
from transformers import AutoTokenizer

RANDOM_SEED = 21
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Load data and build Features

In [ ]:
def extract_missing_features(df: pl.DataFrame, name:str):
    batch_group = []
    group_indx = 0
    i = 0

    # Since batch group was not created during generation
    # we need to create for proper grouping in later operations
    while True:
        batch = df['inference_batch_size'][i]
        if batch == 0:
            batch_group_names = [f"{name}_{group_indx}_Failed"] * 1
            i += 1
        else:
            batch_group_names = [f"{name}_{group_indx}"] * batch
            group_indx += 1
            i += batch
        batch_group.extend(batch_group_names)
        if i >= len(df):
            break
    
    assert len(batch_group) == len(df)
    assert df['first_total_tokens'].n_unique() == 1 and df['first_batch_size'].n_unique() == 1

    df = df.with_columns(
        batch_group = pl.Series(batch_group),
        text_word_length_tmp = pl.col('texts').str.split(' ').list.len(),
        text_char_length_tmp = pl.col('texts').str.len_chars(),
    )

    first_text = df.filter(pl.col("gpu_memory_usage") > 0)['texts'][0]
    first_token_length = df.filter(pl.col("gpu_memory_usage") > 0)['token_length'][0]
    first_batch_group = df.filter(pl.col("gpu_memory_usage") > 0)['batch_group'][0]

    # make sure the first token length is the largest token length
    # this happens when the df is not sorted or saved in the correct order
    # ideally, the df should have executed the biggest token first
    if first_token_length != df['first_top_token_size'][0]:
        first_df = df.filter(
            (pl.col('gpu_memory_usage') == pl.col('first_gpu_memory_peak_usage')) &
            (pl.col('total_tokens') == pl.col('first_total_tokens')) &
            (pl.col('inference_batch_size') == pl.col('first_batch_size')) &
            (pl.col('top_token_size') == pl.col('first_top_token_size')) 
        )
        first_text = first_df['texts'][0]
        first_token_length = first_df['token_length'][0]
        first_batch_group = first_df['batch_group'][0]
        assert first_token_length == df['first_top_token_size'][0]
    
    else:
        # make sure the first token length is the largest token length for if the df is already sorted
        assert first_token_length >= df['token_length'].max() 

    first_text_word_length = len(first_text.split(' '))
    first_text_char_length = len(first_text)

    df = df.with_columns(
        text_word_length_tmp = pl.when(pl.col('batch_group') == first_batch_group).then(pl.lit(first_text_word_length)).otherwise(pl.col('text_word_length_tmp')),
        text_char_length_tmp = pl.when(pl.col('batch_group') == first_batch_group).then(pl.lit(first_text_char_length)).otherwise(pl.col('text_char_length_tmp')),
    )
    
    # add some stats features
    df = df.with_columns(
        token_mean_y = pl.col('token_length').mean().over('batch_group'),
        token_std_y = pl.col('token_length').std().over('batch_group'),
        token_sum_y = pl.col('token_length').sum().over('batch_group'),
        token_max_y = pl.col('token_length').max().over('batch_group'), 
        token_min_y = pl.col('token_length').min().over('batch_group'),
        token_median_y = pl.col('token_length').median().over('batch_group').cast(pl.Int64),
        token_mode_y = pl.col('token_length').mode().first().over('batch_group').cast(pl.Int64), 

        token_mean_x = pl.col('token_length').filter(pl.col('batch_group') == first_batch_group).mean(),
        token_std_x = pl.col('token_length').filter(pl.col('batch_group') == first_batch_group).std(),
        token_sum_x = pl.col('token_length').filter(pl.col('batch_group') == first_batch_group).sum(),
        token_max_x = pl.col('token_length').filter(pl.col('batch_group') == first_batch_group).max(),
        token_min_x = pl.col('token_length').filter(pl.col('batch_group') == first_batch_group).min(),
        token_median_x = pl.col('token_length').filter(pl.col('batch_group') == first_batch_group).median().cast(pl.Int64),
        token_mode_x = pl.col('token_length').filter(pl.col('batch_group') == first_batch_group).mode().first().cast(pl.Int64),

        word_mean_y = pl.col('text_word_length_tmp').mean().over('batch_group'),
        word_std_y = pl.col('text_word_length_tmp').std().over('batch_group'),
        word_sum_y = pl.col('text_word_length_tmp').sum().over('batch_group'),
        word_max_y = pl.col('text_word_length_tmp').max().over('batch_group'),
        word_min_y = pl.col('text_word_length_tmp').min().over('batch_group'),
        word_median_y = pl.col('text_word_length_tmp').median().over('batch_group').cast(pl.Int64),
        word_mode_y = pl.col('text_word_length_tmp').mode().first().over('batch_group').cast(pl.Int64),

        word_mean_x = pl.col('text_word_length_tmp').filter(pl.col('batch_group') == first_batch_group).mean(),
        word_std_x = pl.col('text_word_length_tmp').filter(pl.col('batch_group') == first_batch_group).std(),
        word_sum_x = pl.col('text_word_length_tmp').filter(pl.col('batch_group') == first_batch_group).sum(),
        word_max_x = pl.col('text_word_length_tmp').filter(pl.col('batch_group') == first_batch_group).max(),
        word_min_x = pl.col('text_word_length_tmp').filter(pl.col('batch_group') == first_batch_group).min(),
        word_median_x = pl.col('text_word_length_tmp').filter(pl.col('batch_group') == first_batch_group).median().cast(pl.Int64),
        word_mode_x = pl.col('text_word_length_tmp').filter(pl.col('batch_group') == first_batch_group).mode().first().cast(pl.Int64),

        char_mean_y = pl.col('text_char_length_tmp').mean().over('batch_group'),
        char_std_y = pl.col('text_char_length_tmp').std().over('batch_group'),
        char_sum_y = pl.col('text_char_length_tmp').sum().over('batch_group'),
        char_max_y = pl.col('text_char_length_tmp').max().over('batch_group'),
        char_min_y = pl.col('text_char_length_tmp').min().over('batch_group'),
        char_median_y = pl.col('text_char_length_tmp').median().over('batch_group').cast(pl.Int64),
        char_mode_y = pl.col('text_char_length_tmp').mode().first().over('batch_group').cast(pl.Int64),

        char_mean_x = pl.col('text_char_length_tmp').filter(pl.col('batch_group') == first_batch_group).mean(),
        char_std_x = pl.col('text_char_length_tmp').filter(pl.col('batch_group') == first_batch_group).std(),
        char_sum_x = pl.col('text_char_length_tmp').filter(pl.col('batch_group') == first_batch_group).sum(),
        char_max_x = pl.col('text_char_length_tmp').filter(pl.col('batch_group') == first_batch_group).max(),
        char_min_x = pl.col('text_char_length_tmp').filter(pl.col('batch_group') == first_batch_group).min(),
        char_median_x = pl.col('text_char_length_tmp').filter(pl.col('batch_group') == first_batch_group).median().cast(pl.Int64),
        char_mode_x = pl.col('text_char_length_tmp').filter(pl.col('batch_group') == first_batch_group).mode().first().cast(pl.Int64),

        first_text = pl.lit(first_text),
        first_token_length = pl.lit(first_token_length),
        first_text_word_length = pl.lit(first_text_word_length),
        first_text_char_length = pl.lit(first_text_char_length),
    )

    return df

def get_diffs(df):
    df = df.with_columns(
        token_max_diff = pl.col("token_max_y") / pl.col("token_max_x"),
        batch_size_diff = pl.col("inference_batch_size") / pl.col("first_batch_size"),
        total_paddings_diff = pl.col("total_paddings") / pl.col("first_total_paddings"),
        word_max_diff = pl.col("word_max_y") / pl.col("word_max_x"),
        char_max_diff = pl.col("char_max_y") / pl.col("char_max_x"),

        token_mean_diff = pl.col("token_mean_y") / pl.col("token_mean_x"),
        token_std_diff = pl.col("token_std_y") / pl.col("token_std_x"),
        token_min_diff = pl.col("token_min_y") / pl.col("token_min_x"),
        token_median_diff = pl.col("token_median_y") / pl.col("token_median_x"),
        token_mode_diff = pl.col("token_mode_y") / pl.col("token_mode_x"),
        token_sum_diff = pl.col("token_sum_y") / pl.col("token_sum_x"),

        word_mean_diff = pl.col("word_mean_y") / pl.col("word_mean_x"),
        word_std_diff = pl.col("word_std_y") / pl.col("word_std_x"),
        word_min_diff = pl.col("word_min_y") / pl.col("word_min_x"),
        word_median_diff = pl.col("word_median_y") / pl.col("word_median_x"),
        word_mode_diff = pl.col("word_mode_y") / pl.col("word_mode_x"),
        word_sum_diff = pl.col("word_sum_y") / pl.col("word_sum_x"),

        char_mean_diff = pl.col("char_mean_y") / pl.col("char_mean_x"),
        char_std_diff = pl.col("char_std_y") / pl.col("char_std_x"),
        char_min_diff = pl.col("char_min_y") / pl.col("char_min_x"),
        char_median_diff = pl.col("char_median_y") / pl.col("char_median_x"),
        char_mode_diff = pl.col("char_mode_y") / pl.col("char_mode_x"),
        char_sum_diff = pl.col("char_sum_y") / pl.col("char_sum_x"),

    ).fill_nan(0).fill_null(0).with_columns(
        total_paddings_diff = (
            pl.when(pl.col("total_paddings_diff").is_infinite())
            .then(0)
            .otherwise(pl.col("total_paddings_diff"))
        )
    )
    return df

In [ ]:
dfs = []
data_paths = [
    "train_regressor/data/training_data/v1",
    "train_regressor/data/training_data/v2"
]

rename_cols = {
    "first_batch_size": "batch_size_x",
    "inference_batch_size": "batch_size_y"
}

for data_path in data_paths:
    for file in tqdm(sorted(os.listdir(data_path)), desc="Processing files"):
        if not file.endswith(".parquet"):
            continue
        file_path = os.path.join(data_path, file)
        df = pl.read_parquet(file_path)
        df = extract_missing_features(df, file)
        df = get_diffs(df)
        dfs.append(df)

df = pl.concat(dfs)

# Remove failed batches
df = df.filter(pl.col("gpu_memory_usage") > 0)

# Calculate the difference between the current and first batch size, which will be our target
df = df.with_columns(
    memory_usage_diff = pl.col("gpu_memory_usage") / pl.col("first_gpu_memory_peak_usage"),
)
df = df.rename(rename_cols)

# Remove duplicates
unique_features_to_use = [
    "batch_size_x", "batch_size_y", "memory_usage_diff",
    'token_length', 'text_word_length_tmp',
]
print("Before unique: ", len(df))
df = df.unique(
    subset=unique_features_to_use,
    keep="first",
    maintain_order=True,
).sort(unique_features_to_use)
df = df.select(sorted(df.columns))
print("After unique: ", len(df))

In [ ]:
# Check the distribution of the memory_usage_diff
above_09 = len(df.filter(pl.col("memory_usage_diff") > 0.9))
above_09_percent = above_09 / len(df)
below_09 = len(df.filter(pl.col("memory_usage_diff") <= 0.9))
below_09_percent = below_09 / len(df)

print(f"Above 0.9: {above_09_percent:.2%} | {above_09}")
print(f"Below 0.9: {below_09_percent:.2%} | {below_09}")

## Create Group Columns
Will be used for stratify split 

In [ ]:
from copy import copy

def add_group_cols(df, max_count_per_group=1500):
    custom_breaks = [round(df["memory_usage_diff"].min(), 4), round(df["memory_usage_diff"].max(), 4)]
    
    while True:
        cb_copy = copy(custom_breaks)
        bucket_labels = []
        for i in range(len(cb_copy) + 1):
            is_last = i == len(cb_copy) 
            bucket_label = f"bucket_{cb_copy[i]}" if not is_last else "bucket_10"
            bucket_labels.append(bucket_label)

        df = df.with_columns(
            group_1 = pl.when(pl.col('memory_usage_diff') > 0.9).then(1).otherwise(0),
            group_2 = pl.col("memory_usage_diff").cut(
                breaks=cb_copy, 
                labels=bucket_labels)
        )
        group_2_cols_and_counts = df.group_by("group_2").count().sort("count", descending=True)
        top_count = group_2_cols_and_counts["count"][0]

        if top_count < max_count_per_group:
            break

        group_break_point = float(group_2_cols_and_counts["group_2"][0].split("_")[-1])
        group_break_point_behind = custom_breaks[custom_breaks.index(group_break_point) - 1]
        new_break_point = ((group_break_point - group_break_point_behind) / 2) + group_break_point_behind
        new_break_point = round(new_break_point, 5)
        custom_breaks.append(new_break_point)
        custom_breaks = sorted(custom_breaks)

    print(group_2_cols_and_counts) # to see the distribution
    return df, ["group_1", "group_2"]

df, group_cols= add_group_cols(df)

## Training

In [ ]:
def prepare_data(df, cols_to_use, target_col, group_cols, split_random_state=RANDOM_SEED, test_size=0.1,):
    X = df.select(cols_to_use + group_cols).to_pandas()
    y = df.select(target_col).to_pandas()
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=split_random_state,
        stratify=X[group_cols],
    )
    X_train = X_train.drop(columns=group_cols)
    X_val = X_val.drop(columns=group_cols) 
    return X_train, X_val, y_train, y_val


def train_model(X_train, y_train, X_val, y_val, training_args, weight_ranges, monotone_constraints):
    weights_train = np.ones(len(y_train))
    weights_val = np.ones(len(y_val))

    for start_range, end_range, value in weight_ranges:
        weights_train[((y_train >= start_range) & (y_train < end_range)).to_numpy().squeeze()] = value
        weights_val[((y_val >= start_range) & (y_val < end_range)).to_numpy().squeeze()] = value

    final_monotone_constraints = []
    for col in X_train.columns:
        if col in monotone_constraints:
            final_monotone_constraints.append(monotone_constraints[col])
        else:
            final_monotone_constraints.append(0)

    # Train with weights
    model = xgb.XGBRegressor(
        n_estimators=training_args["n_estimators"],
        learning_rate=training_args["learning_rate"],
        max_depth=training_args["max_depth"],
        objective=training_args["objective"],
        random_state=training_args["random_state"],
        n_jobs=training_args["n_jobs"],
        subsample=1.0,
        colsample_bytree=1.0,
        monotone_constraints=tuple(final_monotone_constraints),
    )

    model.fit(
        X_train, y_train,
        sample_weight=weights_train,
        eval_set=[(X_val, y_val)],
        sample_weight_eval_set=[weights_val],
        verbose=training_args["n_estimators"]//5,
    )
    return model


def visualize_residuals(y_val_critical, y_pred_critical, start_x, end_x):
    plt.scatter(y_val_critical, y_pred_critical, alpha=0.5)
    plt.plot([start_x, end_x], [start_x, end_x], color='red', linestyle='--') # Perfect prediction line
    plt.xlabel('Actual y')
    plt.ylabel('Predicted y')
    plt.title('Critical Region: Actual vs Predicted')
    plt.show()


def accuracy_will_not_OOM(y_pred_critical, y_val_critical):
    """
    Accuracy given the model suggested batch size will not OOM.
    y_val_critical (Ground truth) > 1.0 but the y_pred_critical (Model's prediction) < 1.0, it will OOM.
    So calculate the accuracy that the model suggested batch size will not OOM.
    """
    over_estimates_count = ((y_val_critical > 1.0) & (y_pred_critical <= 1.0)).sum()
    oom_accuracy = 1 - (over_estimates_count / len(y_pred_critical))
    print(f"Count of Overestimates: {over_estimates_count} \t| Total: {len(y_pred_critical)}")
    print("OOM Accuracy: ", oom_accuracy)


def evaluate_model(model, X_val, y_val, critical_range_start, critical_range_end):
    # 1. Generate predictions on the validation set
    y_pred_val = model.predict(X_val)

    # 2. Calculate overall baseline metrics
    overall_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    overall_r2 = r2_score(y_val, y_pred_val)

    print("--- Overall Validation Performance ---")
    print(f"RMSE: {overall_rmse:.4f}")
    print(f"R2:   {overall_r2:.4f}")

    # 3. Isolate and evaluate the hyper-critical region 
    mask = ((y_val >= critical_range_start) & (y_val <= critical_range_end)).to_numpy().squeeze()
    y_val_critical = y_val[mask].to_numpy().squeeze()
    y_pred_critical = y_pred_val[mask]

    if len(y_val_critical) > 0:
            critical_rmse = np.sqrt(mean_squared_error(y_val_critical, y_pred_critical))
            critical_mae = mean_absolute_error(y_val_critical, y_pred_critical)
            
            # Mean Absolute Percentage Error (MAPE) to check your 1% margin goal
            critical_mape = np.mean(np.abs((y_val_critical - y_pred_critical) / y_val_critical)) * 100
            
            print(f"--- Critical Region [{critical_range_start}, {critical_range_end}] ---")
            print(f"Samples evaluated: {len(y_val_critical)}")
            print(f"RMSE: {critical_rmse:.4f}")
            print(f"MAE:  {critical_mae:.4f}")
            print(f"MAPE: {critical_mape:.2f}%")
    else:
            print("WARNING: No validation samples in the [0.8, 1.0] range.")

    accuracy_will_not_OOM(y_pred_critical, y_val_critical)
    
    return y_pred_val, y_val_critical, y_pred_critical


def train(cols_to_use, target_col, df, training_args, monotone_constraints, weight_ranges, group_cols=["group_1", "group_2"], critical_range_start=0.5, critical_range_end=1.1):
    X_train, X_val, y_train, y_val = prepare_data(df, cols_to_use, target_col, group_cols, training_args["random_state"])
    model = train_model(X_train, y_train, X_val, y_val, training_args, weight_ranges, monotone_constraints)
    y_pred_val, y_val_critical, y_pred_critical = evaluate_model(model, X_val, y_val, critical_range_start, critical_range_end)
    # visualize_residuals(y_val_critical, y_pred_critical, critical_range_start, critical_range_end)
    return model, y_pred_val, y_val_critical, y_pred_critical

In [ ]:
#### Training configs

# This features are picked after multiple experiments
wanted_features_col = [
    "batch_size_x", "batch_size_y", "batch_size_diff",
    "token_max_diff", "token_mean_diff", "token_sum_diff", 
    "word_max_diff", "word_mean_diff", "word_sum_diff",
    "char_sum_diff",
]

# Enforce monotonicity
monotone_constraints = {
    # "batch_size_diff": 1, 
    # "token_sum_diff" : 1,
}

for col in wanted_features_col:
    assert not df.select(pl.col(col).is_infinite().any()).item(), f"Infinite values in {col}"
    assert not df.select(pl.col(col).is_nan().any()).item(), f"NaN values in {col}"
    assert not df.select(pl.col(col).is_null().any()).item(), f"Null values in {col}"


critical_range_start = 0.0
critical_range_end = 1.2

weight_ranges = [
    # (startrange, end range, value)
    (0.00, 0.10, 4.0),
    (0.10, 0.25, 3.4),
    (0.25, 0.40, 2.8),
    (0.40, 0.60, 2.2),
    (0.60, 0.85, 2.0),
    (0.85, 1.30, 1.7),
    (1.30, 2.00, 1.5),
    (2.00, 3.00, 1.2),
]

training_args = {
    "n_estimators": 2000,
    "learning_rate": 0.15,
    "max_depth": 12,
    "objective": "reg:squarederror",
    "random_state": RANDOM_SEED,
    "n_jobs": 1,
}

In [ ]:
#### Train ####
model, y_pred_val, y_val_critical, y_pred_critical = train(
    wanted_features_col, 
    "memory_usage_diff", 
    df, 
    training_args,
    monotone_constraints,
    weight_ranges,
    group_cols,
    critical_range_start, 
    critical_range_end,
    )

## Predict on Examples
(Simulate production environment)

In [ ]:
# load dataset, tokenizer and sampler -> Model Input Data
ds = load_dataset("OpenAssistant/oasst1")
df_en = ds['train'].to_polars().filter(pl.col('lang') == 'en')
model_name = "facebook/nllb-200-distilled-600M"
source_lang_code = "eng_Latn"
target_lang_code = "fra_Latn"

n_variations = 10
maximum_ranges = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
max_length = [128, 256, 512]
batch_size = [1, 2, 8, 64, 128]

nllb_tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang=source_lang_code, tgt_lang=target_lang_code)
sampler_input_features = []
sampler_configs = []

for i in range(n_variations):
    texts_data = df_en.sample(1000, seed=i)['text'].to_list()
    m_range = random.choice(maximum_ranges)
    m_length = random.choice(max_length)
    b_size = random.choice(batch_size)
    sampler = dynabatch_sampler(
        texts_data,
        tokenizer=nllb_tokenizer,
        batch_size=b_size,
        max_input_token_length=m_length,
        max_batch_range=m_range,
        save_input_features=True,
    )
    sampler_input_features.append(sampler.input_features)
    sampler_configs.append(
        {
            "batch_size": b_size,
            "max_input_token_length": m_length,
            "max_batch_range": m_range,
        }
    )

In [ ]:
n_samples_per_batch = 2

def get_prod_regressor():
    _regressor = xgb.XGBRegressor()
    _regressor.load_model("/home/ben/work/dynabatch/dynabatch/models/regressor.ubj")
    return _regressor
prod_model = get_prod_regressor()

def add_all_possible_diffs(df, wanted_cols):
    df = pl.from_pandas(df)
    all_cols = df.columns
    for col in wanted_cols:
        if col.endswith("_diff"):
                if col.replace("_diff", "_x") in all_cols \
                    and col.replace("_diff", "_y") in all_cols:
                    df = df.with_columns(
                        (pl.col(col.replace("_diff", "_y")) / pl.col(col.replace("_diff", "_x"))).alias(col)
                    )
                else:
                    print(f"{col.replace("_diff", "_x")}: {col.replace("_diff", "_x") in all_cols}")
                    print(f"{col.replace("_diff", "_y")}: {col.replace("_diff", "_y") in all_cols}")
    return df.to_pandas()

def print_list(list, max_len_per_line=8):
    for i, item in enumerate(list):
        if i % max_len_per_line == 0:
            print()
        print(item, end=" \t| ")
    print()


def predict(input_features):
    input_features = add_all_possible_diffs(input_features, wanted_features_col)
    wanted_train_cols_new = input_features[model.get_booster().feature_names]
    wanted_train_cols_old = input_features[prod_model.get_booster().feature_names]
    
    y_pred_new = np.round(model.predict(wanted_train_cols_new), 2)
    y_pred_old = np.round(prod_model.predict(wanted_train_cols_old), 2)

    combined_pred_and_token_sum_diff = []
    token_sum_diff = wanted_train_cols_new['token_sum_diff'].to_numpy()

    for i in range(len(y_pred_new)):
        combined_pred_and_token_sum_diff.append([round(y_pred_old[i].item(), 2), round(y_pred_new[i].item(), 2), round(token_sum_diff[i].item(), 2)])
    print_list(combined_pred_and_token_sum_diff)


count = 0
# NEW_MODEL vs PROD_MODEL vs Token Sum Diff 
# Using Token Sum Diff because we dont the GT, and Token Sum Diff is the closest proxy
# What we looking here is if the model prediction is not too far off from the Token Sum Diff and
# the prediction monotonicty is somewhat maintained when the Token Sum Diff is increasing
print("[Old Model, New Model, Token Sum Diff], .. | ...")
for batches, config in zip(sampler_input_features, sampler_configs):
    indices = sorted(random.sample(range(len(batches)), n_samples_per_batch))
    selected_batches = [batches[i] for i in indices]
    print(f"\n###################### Batch Size: {config['batch_size']} | Max Length: {config['max_input_token_length']} | Max Range: {config['max_batch_range']} ######################")
    for batch in selected_batches:
        print(f"------------------------- {count} ---------------------------")
        predict(batch)
        count += 1


In [ ]:
# Save model
model.save_model("dynabatch/models/regressor.ubj")